In [1]:
!pip install -U transformers
!pip install datasets pandas openpyxl

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.2/40.2 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 28.4 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 4.52.2
    Uninstalling transformers-4.52.2:
      Successfully uninstalled transformers-4.52.2


In [2]:
# 1. 라이브러리 불러오기
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import DataCollatorForLanguageModeling, TrainingArguments, Trainer

In [3]:
# 2. 데이터 불러오기 및 전처리
dfs = []
for i in range(1, 31):
    file_name = f"drive/MyDrive/2025인공지능6팀/Lcms/LcmsExcelView ({i}).xlsx"
    try:
        tempdf = pd.read_excel(file_name, engine='openpyxl', header=2,
                               usecols=['원 형태', '교정 형태', '앞 문맥', '중심어', '뒤 문맥'])
        dfs.append(tempdf)
    except Exception as e:
        print(f"[{i}] 파일 오류: {file_name} → {type(e).__name__}: {e}")

df = pd.concat(dfs, ignore_index=True)
df = df.fillna('')
df["원 형태"] = df["원 형태"].astype(str)
df["교정 형태"] = df["교정 형태"].astype(str)

# 앞뒤 문맥 키로 그룹핑
df["문맥키"] = df["앞 문맥"].str.strip() + "||" + df["뒤 문맥"].str.strip()
grouped = df.groupby("문맥키").agg({
    "원 형태": lambda x: ''.join(x),
    "교정 형태": lambda x: ''.join(x)
}).reset_index()

grouped = grouped.rename(columns={"원 형태": "input", "교정 형태": "target"})
grouped.drop(columns=["문맥키"], inplace=True)
grouped = grouped.drop_duplicates()
print(grouped)

/usr/local/lib/python3.11/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/usr/local/lib/python3.11/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/usr/local/lib/python3.11/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/usr/local/lib/python3.11/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/usr/local/lib/python3.11/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default sty

     input target
0       꼬뿌     공부
1       셔로     서로
2     프라이패   프라이팬
3       밴드     밴드
4       바브     바쁘
...    ...    ...
2643    활려     화려
2644     도      도
2646   공슈리    공휴일
2647   펵돌은    벽돌을
2649    개원     개월

[1773 rows x 2 columns]


/usr/local/lib/python3.11/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [4]:
# 3. KoGPT2 모델과 토크나이저 불러오기
model_name = "skt/kogpt2-base-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
# 패딩 토큰 지정
tokenizer.pad_token = tokenizer.eos_token

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.00k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.83M [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/513M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/513M [00:00<?, ?B/s]

In [5]:
# 4. 학습용 데이터셋 생성
dataset = Dataset.from_pandas(grouped)

def format_example(example):
    return {"text": f"{example['input']}<sep>{example['target']}"}

formatted_dataset = dataset.map(format_example, remove_columns=dataset.column_names)
print(formatted_dataset)

Map:   0%|          | 0/1773 [00:00<?, ? examples/s]

Dataset({
    features: ['text'],
    num_rows: 1773
})


In [6]:
# 5. 토크나이징
def tokenize(example):
    return tokenizer(example["text"], truncation=True, padding="max_length", max_length=64)

tokenized_dataset = formatted_dataset.map(tokenize)



Map:   0%|          | 0/1773 [00:00<?, ? examples/s]

In [14]:
# 패딩 토큰 지정 및 모델 config에 반영
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.pad_token_id
def tokenize(example):
    encodings = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=64,
        return_attention_mask=True
    )
    labels = encodings["input_ids"].copy()
    labels = [-100 if token == tokenizer.pad_token_id else token for token in labels]
    encodings["labels"] = labels
    return encodings
model.resize_token_embeddings(len(tokenizer))

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Embedding(51201, 768)

In [15]:
# 6. 학습/검증 데이터 분리
split = tokenized_dataset.train_test_split(test_size=0.1)
train_dataset = split["train"]
eval_dataset = split["test"]

In [9]:
# 7. 데이터 collator (for causal LM)
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

In [16]:
# 8. 학습 설정
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_steps=20,
    save_total_limit=2,
    save_strategy="epoch",
    logging_dir="./logs",
    report_to="none",  # wandb 등 연결 안 할 경우
)

In [17]:
# 9. Trainer 정의
trainer = Trainer(
    model=model,
    args=training_args,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator
)

<ipython-input-17-f77cc348241a>:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [18]:
max_token_id = max(max(ids) for ids in tokenized_dataset["input_ids"])
vocab_size = tokenizer.vocab_size

print(f"Max token ID: {max_token_id}, Vocab size: {vocab_size}")

Max token ID: 51200, Vocab size: 51200


In [ ]:
# 10. 학습 시작
trainer.train()

`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,2.133100,2.342878
2,1.527800,2.185505
3,1.227900,2.219091
4,0.959800,2.336361
